# Bài 1

In [1]:
%%writefile bai1.py

from pyspark import SparkContext

sc = SparkContext.getOrCreate()

movies_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/movies.txt"
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"

movies = sc.textFile(movies_path)
movies_rdd = movies.map(lambda line: line.split(",")) \
                   .map(lambda x: (x[0], x[1]))

ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# MovieID -> (Rating, 1)
ratings_map = ratings.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[1], (float(x[2]), 1)))

# Tính tổng điểm và số lượt rating
ratings_reduce = ratings_map.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Tính average rating
ratings_avg = ratings_reduce.mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Join với tên phim
movie_stats = movies_rdd.join(ratings_avg)

# Kết quả
results = movie_stats.collect()
print("ĐIỂM TRUNG BÌNH CỦA PHIM")

for movie in results:

    movie_id = movie[0]
    title = movie[1][0]
    avg_rating = movie[1][1][0]
    total_count = movie[1][1][1]

    print(
        f"MovieID: {movie_id}  "
        f"Title: {title}  "
        f"Average Rating: {avg_rating:.2f}  "
        f"Total Ratings: {total_count}"
    )

# Phim rating cao nhất
top_movie = movie_stats.filter(
    lambda x: x[1][1][1] >= 5
).takeOrdered(
    1,
    key=lambda x: -x[1][1][0]
)

print("\nPHIM CÓ ĐIỂM CAO NHẤT")

for movie in top_movie:

    print(f"MovieID: {movie[0]}")
    print(f"Title: {movie[1][0]}")
    print(f"Average Rating: {movie[1][1][0]:.2f}")
    print(f"Total Ratings: {movie[1][1][1]}")

with open("output_bai1.txt", "w", encoding="utf-8") as f:

    f.write("ĐIỂM TRUNG BÌNH VÀ SỐ LƯỢT ĐÁNH GIÁ\n\n")

    for movie in results:

        movie_id = movie[0]
        title = movie[1][0]
        avg_rating = movie[1][1][0]
        total_count = movie[1][1][1]

        f.write(
            f"MovieID: {movie_id}  "
            f"Title: {title}  "
            f"Average Rating: {avg_rating:.2f}  "
            f"Total Ratings: {total_count}\n"
        )

    f.write("\nPHIM CÓ ĐIỂM TRUNG BÌNH CAO NHẤT\n\n")

    for movie in top_movie:

        f.write(f"MovieID: {movie[0]}\n")
        f.write(f"Title: {movie[1][0]}\n")
        f.write(f"Average Rating: {movie[1][1][0]:.2f}\n")
        f.write(f"Total Ratings: {movie[1][1][1]}\n")

print("Đã lưu kết quả vào output_bai1.txt")

Writing bai1.py


In [2]:
!python bai1.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:26:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
ĐIỂM TRUNG BÌNH CỦA PHIM                                                        
MovieID: 1015  Title: Sunset Boulevard (1950)  Average Rating: 4.36  Total Ratings: 7
MovieID: 1037  Title: The Lord of the Rings: The Fellowship of the Ring (2001)  Average Rating: 3.89  Total Ratings: 18
MovieID: 1043  Title: No Country for Old Men (2007)  Average Rating: 3.89  Total Ratings: 18
MovieID: 1050  Title: Mad Max: Fury Road (2015)  Average Rating: 3.47  Total Ratings: 18
MovieID: 1028  Title: Fight Club (1999)  Average Rating: 3.50  Total Ratings: 7
MovieID: 1010  Title: Lawrence of Arabia (1962)  Average Rating: 3.44  Total Ratings: 18
MovieID:

# Bài 2

In [3]:
%%writefile bai2.py

from pyspark import SparkContext

sc = SparkContext.getOrCreate()

movies_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/movies.txt"
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"

movies = sc.textFile(movies_path)

# MovieID -> List Genres
movies_rdd = movies.map(lambda line: line.split(",")) \
                   .map(lambda x: (x[0], x[2].split("|")))

# ratings
ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# MovieID -> Rating
ratings_rdd = ratings.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[1], float(x[2])))

# Join MovieID
joined_rdd = ratings_rdd.join(movies_rdd)

# Genre -> (Rating,1)
genre_rating = joined_rdd.flatMap(
    lambda x: [(genre, (x[1][0], 1)) for genre in x[1][1]]
)

# Tính tổng điểm và số lượt
genre_reduce = genre_rating.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Tính điểm trung bình
genre_avg = genre_reduce.mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Kết quả
results = genre_avg.sortBy(
    lambda x: x[1][0],
    ascending=False
).collect()
print(" ĐIỂM TRUNG BÌNH THEO THỂ LOẠI ")

for genre in results:

    genre_name = genre[0]
    avg_rating = genre[1][0]
    total_count = genre[1][1]

    print(
        f"Genre: {genre_name}  "
        f"Average Rating: {avg_rating:.2f}  "
    )

# lưu output
with open("output_bai2.txt", "w", encoding="utf-8") as f:

    f.write(" ĐIỂM TRUNG BÌNH THEO THỂ LOẠI \n\n")

    for genre in results:

        genre_name = genre[0]
        avg_rating = genre[1][0]
        total_count = genre[1][1]

        f.write(
            f"Genre: {genre_name} "
            f"Average Rating: {avg_rating:.2f}\n"
        )

print("Đã lưu kết quả vào output_bai2.txt")

Writing bai2.py


In [4]:
!python bai2.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:27:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
 ĐIỂM TRUNG BÌNH THEO THỂ LOẠI                                                  
Genre: Film-Noir  Average Rating: 4.36  
Genre: Horror  Average Rating: 4.00  
Genre: Mystery  Average Rating: 4.00  
Genre: Fantasy  Average Rating: 3.86  
Genre: Crime  Average Rating: 3.81  
Genre: Drama  Average Rating: 3.76  
Genre: Sci-Fi  Average Rating: 3.73  
Genre: Action  Average Rating: 3.71  
Genre: Thriller  Average Rating: 3.70  
Genre: Family  Average Rating: 3.67  
Genre: Adventure  Average Rating: 3.63  
Genre: Biography  Average Rating: 3.56  
Đã lưu kết quả vào output_bai2.txt


# Bài 3

In [5]:
%%writefile bai3.py

from pyspark import SparkContext

sc = SparkContext.getOrCreate()

movies_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/movies.txt"
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"
users_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/users.txt"

movies = sc.textFile(movies_path)

# MovieID -> Title
movies_rdd = movies.map(lambda line: line.split(",")) \
                   .map(lambda x: (x[0], x[1]))

# users
users = sc.textFile(users_path)

# UserID -> Gender
users_rdd = users.map(lambda line: line.split(",")) \
                 .map(lambda x: (x[0], x[1]))

# ratings
ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# UserID -> (MovieID, Rating)
ratings_rdd = ratings.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[0], (x[1], float(x[2]))))

# Join ratings với users
joined_rdd = ratings_rdd.join(users_rdd)

# ((MovieID, Gender) -> (Rating,1))
movie_gender = joined_rdd.map(
    lambda x: (
        (x[1][0][0], x[1][1]),
        (x[1][0][1], 1)
    )
)

# Tính tổng điểm và số lượt
reduce_rdd = movie_gender.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Tính average rating
avg_rdd = reduce_rdd.mapValues(
    lambda x: x[0] / x[1]
)

# Đổi format để join tên phim
movie_avg = avg_rdd.map(
    lambda x: (
        x[0][0],
        (x[0][1], x[1])
    )
)

# Join với movie title
final_rdd = movies_rdd.join(movie_avg)

# Kết quả
results = final_rdd.collect()

print(" ĐIỂM TRUNG BÌNH PHIM THEO GIỚI TÍNH ")

for item in results:

    movie_id = item[0]
    title = item[1][0]
    gender = item[1][1][0]
    avg_rating = item[1][1][1]

    print(
        f"MovieID: {movie_id}  "
        f"Title: {title}  "
        f"Gender: {gender}  "
        f"Average Rating: {avg_rating:.2f}"
    )

# Lưu file txt
with open("output_bai3.txt", "w", encoding="utf-8") as f:

    f.write(" ĐIỂM TRUNG BÌNH PHIM THEO GIỚI TÍNH \n\n")

    for item in results:

        movie_id = item[0]
        title = item[1][0]
        gender = item[1][1][0]
        avg_rating = item[1][1][1]

        f.write(
            f"MovieID: {movie_id}  "
            f"Title: {title}  "
            f"Gender: {gender}  "
            f"Average Rating: {avg_rating:.2f}\n"
        )

print("Đã lưu kết quả vào output_bai3.txt")

Writing bai3.py


In [6]:
!python bai3.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:27:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
 ĐIỂM TRUNG BÌNH PHIM THEO GIỚI TÍNH                                            
MovieID: 1040  Title: Gladiator (2000)  Gender: M  Average Rating: 3.59
MovieID: 1040  Title: Gladiator (2000)  Gender: F  Average Rating: 3.64
MovieID: 1015  Title: Sunset Boulevard (1950)  Gender: M  Average Rating: 4.33
MovieID: 1015  Title: Sunset Boulevard (1950)  Gender: F  Average Rating: 4.50
MovieID: 1020  Title: E.T. the Extra-Terrestrial (1982)  Gender: M  Average Rating: 3.81
MovieID: 1020  Title: E.T. the Extra-Terrestrial (1982)  Gender: F  Average Rating: 3.55
MovieID: 1047  Title: The Social Network (2010)  Gender: M  Average Rating: 4.00
Movi

# Bài 4

In [7]:
%%writefile bai4.py

from pyspark import SparkContext

sc = SparkContext.getOrCreate()

movies_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/movies.txt"
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"
users_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/users.txt"

# Hàm phân nhóm tuổi
def age_group(age):

    age = int(age)

    if age < 18:
        return "Under 18"

    elif age <= 25:
        return "18-25"

    elif age <= 35:
        return "26-35"

    elif age <= 45:
        return "36-45"

    else:
        return "46+"

# Đọc movies
movies = sc.textFile(movies_path)

# MovieID -> Title
movies_rdd = movies.map(lambda line: line.split(",")) \
                   .map(lambda x: (x[0], x[1]))

# Đọc users
users = sc.textFile(users_path)

# UserID -> Age Group
users_rdd = users.map(lambda line: line.split(",")) \
                 .map(lambda x: (x[0], age_group(x[2])))

# Đọc ratings
ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# UserID -> (MovieID, Rating)
ratings_rdd = ratings.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[0], (x[1], float(x[2]))))

# Join ratings với users
joined_rdd = ratings_rdd.join(users_rdd)

# ((MovieID, AgeGroup) -> (Rating,1))
movie_age = joined_rdd.map(
    lambda x: (
        (x[1][0][0], x[1][1]),
        (x[1][0][1], 1)
    )
)

# Tính tổng điểm và số lượt
reduce_rdd = movie_age.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Tính average rating
avg_rdd = reduce_rdd.mapValues(
    lambda x: x[0] / x[1]
)

# Đổi format để join title
movie_avg = avg_rdd.map(
    lambda x: (
        x[0][0],
        (x[0][1], x[1])
    )
)

# Join với tên phim
final_rdd = movies_rdd.join(movie_avg)

# Kết quả
results = final_rdd.collect()

print(" ĐIỂM TRUNG BÌNH PHIM THEO NHÓM TUỔI ")

for item in results:

    movie_id = item[0]
    title = item[1][0]
    age_group_name = item[1][1][0]
    avg_rating = item[1][1][1]

    print(
        f"MovieID: {movie_id}  "
        f"Title: {title}  "
        f"Age Group: {age_group_name}  "
        f"Average Rating: {avg_rating:.2f}"
    )

# Lưu file txt
with open("output_bai4.txt", "w", encoding="utf-8") as f:

    f.write(" ĐIỂM TRUNG BÌNH PHIM THEO NHÓM TUỔI \n\n")

    for item in results:

        movie_id = item[0]
        title = item[1][0]
        age_group_name = item[1][1][0]
        avg_rating = item[1][1][1]

        f.write(
            f"MovieID: {movie_id}  "
            f"Title: {title}  "
            f"Age Group: {age_group_name}  "
            f"Average Rating: {avg_rating:.2f}\n"
        )

print("Đã lưu kết quả vào output_bai4.txt")

Writing bai4.py


In [8]:
!python bai4.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:27:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
 ĐIỂM TRUNG BÌNH PHIM THEO NHÓM TUỔI                                            
MovieID: 1040  Title: Gladiator (2000)  Age Group: 46+  Average Rating: 3.50
MovieID: 1040  Title: Gladiator (2000)  Age Group: 18-25  Average Rating: 3.50
MovieID: 1040  Title: Gladiator (2000)  Age Group: 36-45  Average Rating: 3.86
MovieID: 1040  Title: Gladiator (2000)  Age Group: 26-35  Average Rating: 3.43
MovieID: 1015  Title: Sunset Boulevard (1950)  Age Group: 36-45  Average Rating: 4.50
MovieID: 1015  Title: Sunset Boulevard (1950)  Age Group: 46+  Average Rating: 4.50
MovieID: 1015  Title: Sunset Boulevard (1950)  Age Group: 18-25  Average Rating: 

# Bài 5

In [9]:
%%writefile bai5.py

from pyspark import SparkContext

sc = SparkContext.getOrCreate()

users_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/users.txt"
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"
occupation_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/occupation.txt"

# occupation
occupation = sc.textFile(occupation_path)

# OccupationID -> OccupationName
occupation_rdd = occupation.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[0], x[1]))

# users
users = sc.textFile(users_path)

# UserID -> OccupationID
users_rdd = users.map(lambda line: line.split(",")) \
                 .map(lambda x: (x[0], x[3]))

# Join users với occupation
# OccupationID -> (UserID, OccupationName)

user_occ = users_rdd.map(
    lambda x: (x[1], x[0])
)

joined_occ = user_occ.join(occupation_rdd)

# UserID -> OccupationName
user_occupation_rdd = joined_occ.map(
    lambda x: (x[1][0], x[1][1])
)

# ratings
ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# UserID -> Rating
ratings_rdd = ratings.map(lambda line: line.split(",")) \
                     .map(lambda x: (x[0], float(x[2])))

# Join ratings với occupation
joined_rdd = ratings_rdd.join(user_occupation_rdd)

# Occupation -> (Rating,1)
occupation_rating = joined_rdd.map(
    lambda x: (
        x[1][1],
        (x[1][0], 1)
    )
)

# Tính tổng điểm và số lượt
reduce_rdd = occupation_rating.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# Tính average rating
avg_rdd = reduce_rdd.mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Kết quả
results = avg_rdd.collect()

print(" ĐÁNH GIÁ THEO OCCUPATION ")

for item in results:

    occupation_name = item[0]
    avg_rating = item[1][0]
    total_ratings = item[1][1]

    print(
        f"Occupation: {occupation_name}  "
        f"Average Rating: {avg_rating:.2f}  "
        f"Total Ratings: {total_ratings}"
    )

# Lưu file txt
with open("output_bai5.txt", "w", encoding="utf-8") as f:

    f.write(" ĐÁNH GIÁ THEO OCCUPATION \n\n")

    for item in results:

        occupation_name = item[0]
        avg_rating = item[1][0]
        total_ratings = item[1][1]

        f.write(
            f"Occupation: {occupation_name}  "
            f"Average Rating: {avg_rating:.2f}  "
            f"Total Ratings: {total_ratings}\n"
        )

print("Đã lưu kết quả vào output_bai5.txt")

Writing bai5.py


In [10]:
!python bai5.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:27:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
 ĐÁNH GIÁ THEO OCCUPATION                                                       
Occupation: Consultant  Average Rating: 3.86  Total Ratings: 14
Occupation: Salesperson  Average Rating: 3.65  Total Ratings: 17
Occupation: Engineer  Average Rating: 3.56  Total Ratings: 18
Occupation: Manager  Average Rating: 3.47  Total Ratings: 16
Occupation: Designer  Average Rating: 4.00  Total Ratings: 13
Occupation: Doctor  Average Rating: 3.69  Total Ratings: 21
Occupation: Journalist  Average Rating: 3.85  Total Ratings: 17
Occupation: Teacher  Average Rating: 3.70  Total Ratings: 5
Occupation: Artist  Average Rating: 3.73  Total Ratings: 11
Occupat

# Bài 6

In [11]:
%%writefile bai6.py

from pyspark import SparkContext
from datetime import datetime

sc = SparkContext.getOrCreate()
ratings1_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_1.txt"
ratings2_path = "/kaggle/input/datasets/kghangco/pyspark-ds200/ratings_2.txt"

# timestamp -> year
def get_year(timestamp):

    return datetime.fromtimestamp(
        int(timestamp)
    ).year

# ratings
ratings1 = sc.textFile(ratings1_path)
ratings2 = sc.textFile(ratings2_path)

ratings = ratings1.union(ratings2)

# Year -> (Rating,1)
ratings_rdd = ratings.map(lambda line: line.split(",")) \
                     .map(
                         lambda x: (
                             get_year(x[3]),
                             (float(x[2]), 1)
                         )
                     )

# Tính tổng điểm và số lượt
reduce_rdd = ratings_rdd.reduceByKey(
    lambda a, b: (
        a[0] + b[0],
        a[1] + b[1]
    )
)

# Tính average rating
avg_rdd = reduce_rdd.mapValues(
    lambda x: (
        x[0] / x[1],
        x[1]
    )
)

# Sắp xếp theo năm
results = avg_rdd.sortByKey().collect()

# kết quả
print(" PHÂN TÍCH ĐÁNH GIÁ THEO THỜI GIAN ")

for item in results:

    year = item[0]
    avg_rating = item[1][0]
    total_ratings = item[1][1]

    print(
        f"Year: {year}  "
        f"Average Rating: {avg_rating:.2f}  "
        f"Total Ratings: {total_ratings}"
    )

# Lưu file txt
with open("output_bai6.txt", "w", encoding="utf-8") as f:

    f.write(" PHÂN TÍCH ĐÁNH GIÁ THEO THỜI GIAN \n\n")

    for item in results:

        year = item[0]
        avg_rating = item[1][0]
        total_ratings = item[1][1]

        f.write(
            f"Year: {year}  "
            f"Average Rating: {avg_rating:.2f}  "
            f"Total Ratings: {total_ratings}\n"
        )

print("Đã lưu kết quả vào output_bai6.txt")

Writing bai6.py


In [12]:
!python bai6.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 08:28:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
 PHÂN TÍCH ĐÁNH GIÁ THEO THỜI GIAN                                              
Year: 2020  Average Rating: 3.75  Total Ratings: 184
Đã lưu kết quả vào output_bai6.txt
26/05/20 08:28:13 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-9e6f7beb-259f-4118-8aef-b706e9319764. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-9e6f7beb-259f-4118-8aef-b706e9319764
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:199)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:116)
	at org.apache.spark.network.util.J